# Generate Storybook Stories for Evaluation (in-place)

In [86]:
import re
import json
from pathlib import Path

### 1. Repo root and configuration

In [87]:
def find_repo_root(marker: str = 'dataset', start: Path | None = None) -> Path:
    """Searches upward from the current working directory until a folder
    named 'marker' is found -- robust against the kernel's working directory
    not matching the notebook's own folder."""
    start = start or Path.cwd()

    for parent in [start, *start.parents]:
        if (parent / marker).is_dir():
            return parent

    raise FileNotFoundError(f"Could not find a folder named '{marker}' above {start}")


REPO_ROOT = find_repo_root()
STORYBOOK_ROOT = REPO_ROOT / 'dataset' / 'storybook'

# 'components' or 'uis'
TYPE = 'components'

CODE_DIR = STORYBOOK_ROOT / 'src' / 'code' / TYPE
STORIES_DIR = STORYBOOK_ROOT / 'src' / 'stories' / TYPE

APPROACHES = ['a', 'b', 'c', 'd']
PROMPT_STRATEGIES = ['few_shot', 'zero_shot']
COMPLEXITIES = ['hard', 'medium', 'simple']

MANIFEST_PATH = REPO_ROOT / 'evaluations' / f'stories_manifest_{TYPE}.json'

BEGIN_MARKER = '// >>> AUTO-GENERATED EVAL STORIES (managed by storybook-generate-stories.ipynb) -- do not edit by hand >>>'
END_MARKER = '// <<< AUTO-GENERATED EVAL STORIES <<<'

print(f'CODE_DIR:    {CODE_DIR}  (exists: {CODE_DIR.exists()})')
print(f'STORIES_DIR: {STORIES_DIR}  (exists: {STORIES_DIR.exists()})')

CODE_DIR:    C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\src\code\components  (exists: True)
STORIES_DIR: C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\src\stories\components  (exists: True)


### 2. Locate existing GT story files and their matching generated variants

In [88]:
def find_gt_story_files() -> list[dict]:
    """Returns [{'story_file': Path, 'complexity': str|None, 'index': str}, ...]"""
    results = []

    if TYPE == 'components':
        for complexity in COMPLEXITIES:
            complexity_dir = STORIES_DIR / complexity

            if not complexity_dir.exists():
                print(f'  Skip (not existing): {complexity_dir}')
                continue

            for story_file in sorted(complexity_dir.glob('*.stories.ts')):
                index = story_file.stem.removesuffix('.stories')
                results.append({'story_file': story_file, 'complexity': complexity, 'index': index})

    else:  # 'uis'
        if not STORIES_DIR.exists():
            print(f'  Skip (not existing): {STORIES_DIR}')
        else:
            for story_file in sorted(STORIES_DIR.glob('*.stories.ts')):
                index = story_file.stem.removesuffix('.stories')
                results.append({'story_file': story_file, 'complexity': None, 'index': index})

    return results


_A_DIAGNOSTICS_PRINTED = set()  # avoid re-printing the same "missing/empty" diagnostic for every mockup


def find_generated_variants(complexity: str | None, index: str) -> list[dict]:
    """Returns every generated .vue file matching this GT story's index (and,
    for 'components', its complexity), across all approaches/prompt strategies
    (and, for 'uis', across both pretty/messy variant subfolders).

    Approach A is rule-based and deterministic -- exactly one output per
    mockup, no model/run variation, and consequently no prompt_strategy
    subfolder in its output layout at all (unlike B/C/D):
      components: code/components/a/{complexity}/{index}-a.vue
      uis:        code/uis/a/{pretty,messy}/{index}.vue   (no '-a' suffix)
    It is handled as its own branch below rather than folding it into the
    prompt_strategy loop used for B/C/D, since that loop assumes a directory
    level that does not exist for A.

    Prints a ONE-TIME diagnostic (not per mockup) if approach 'a's directory
    is missing entirely or exists but yields zero .vue matches for ANY
    mockup -- a plain 'continue' here would otherwise skip A completely
    without any indication of why, which is exactly what made this silent
    for a whole run before this diagnostic was added.

    Returns: [{'vue_file': Path, 'approach': str, 'prompt_strategy': str|None,
               'group': str, 'variant': str|None}, ...]
    """
    matches = []

    for approach in APPROACHES:
        approach_dir = CODE_DIR / approach

        if approach == 'a':
            if not approach_dir.exists():
                if 'a_dir_missing' not in _A_DIAGNOSTICS_PRINTED:
                    print(f'  DIAGNOSTIC: approach \'a\' directory does not exist: {approach_dir} '
                          f'-- Ansatz A will be skipped entirely until this exists.')
                    _A_DIAGNOSTICS_PRINTED.add('a_dir_missing')
                continue

            if TYPE == 'components':
                group_dirs = [approach_dir / complexity] if (approach_dir / complexity).exists() else []
                glob_pattern = f'{index}-a.vue'
            else:  # 'uis'
                group_dirs = [p for p in approach_dir.iterdir() if p.is_dir()]  # pretty/, messy/
                glob_pattern = f'{index}-a.vue'

            if not group_dirs and 'a_no_group_dirs' not in _A_DIAGNOSTICS_PRINTED:
                print(f'  DIAGNOSTIC: approach \'a\' directory {approach_dir} exists but has no '
                      f'matching subfolder for complexity={complexity!r} (components) or no '
                      f'subdirectories at all (uis) -- check the actual folder names on disk.')
                _A_DIAGNOSTICS_PRINTED.add('a_no_group_dirs')

            a_found_any = False
            for group_dir in group_dirs:
                found_here = sorted(group_dir.glob(glob_pattern))

                if not found_here and f'a_empty_{group_dir}' not in _A_DIAGNOSTICS_PRINTED:
                    all_vue = sorted(group_dir.glob('*.vue'))
                    print(f'  DIAGNOSTIC: {group_dir} exists but glob \'{glob_pattern}\' matched 0 files. '
                          f'Files actually present there: {[f.name for f in all_vue] or "(none)"}')
                    _A_DIAGNOSTICS_PRINTED.add(f'a_empty_{group_dir}')

                for vue_file in found_here:
                    a_found_any = True
                    matches.append({
                        'vue_file': vue_file,
                        'approach': approach,
                        'prompt_strategy': None,
                        'group': group_dir.name,
                        'variant': group_dir.name if TYPE == 'uis' else None,
                    })

            continue

        for prompt_strategy in PROMPT_STRATEGIES:
            strategy_dir = approach_dir / prompt_strategy

            if not strategy_dir.exists():
                continue

            if TYPE == 'components':
                group_dirs = [strategy_dir / complexity] if (strategy_dir / complexity).exists() else []
            else:
                group_dirs = [p for p in strategy_dir.iterdir() if p.is_dir()]  # pretty/, messy/

            for group_dir in group_dirs:
                for vue_file in sorted(group_dir.glob(f'{index}-*.vue')):
                    matches.append({
                        'vue_file': vue_file,
                        'approach': approach,
                        'prompt_strategy': prompt_strategy,
                        'group': group_dir.name,
                        'variant': group_dir.name if TYPE == 'uis' else None,
                    })

    return matches

### 2b. Viewport per mockup

In [89]:
GT_FIGMA_DATA_ROOT = REPO_ROOT / 'dataset' / 'figma-data' / 'cleaned' / TYPE

DEFAULT_VIEWPORT = {'width': 1280, 'height': 800}


def _extract_frame_size(node: dict) -> dict | None:
    """Tries several common Figma export field conventions for a frame's
    dimensions, in order of preference. Returns None if none match, so the
    caller can fall back to DEFAULT_VIEWPORT and log that explicitly instead
    of silently guessing."""
    if 'width' in node and 'height' in node:
        return {'width': node['width'], 'height': node['height']}

    bbox = node.get('absoluteBoundingBox')
    if isinstance(bbox, dict) and 'width' in bbox and 'height' in bbox:
        return {'width': bbox['width'], 'height': bbox['height']}

    size = node.get('size')
    if isinstance(size, dict) and 'x' in size and 'y' in size:
        return {'width': size['x'], 'height': size['y']}

    return None


def load_gt_viewport(complexity: str | None, index: str) -> dict:
    """Reads the GT Figma JSON's root frame dimensions for one mockup.
    Falls back to DEFAULT_VIEWPORT (with a warning) if the file is missing
    or none of the known dimension field conventions are found -- verify
    against your actual Figma JSON schema and extend _extract_frame_size()
    if this warning fires for real data."""
    if TYPE == 'components':
        figma_path = GT_FIGMA_DATA_ROOT / complexity / f'{index}.json'
    else:
        figma_path = GT_FIGMA_DATA_ROOT / 'pretty' / f'{index}.json'

    if not figma_path.exists():
        print(f'  WARNING: GT Figma JSON not found at {figma_path} -- using default viewport {DEFAULT_VIEWPORT}')
        return dict(DEFAULT_VIEWPORT)

    root = json.loads(figma_path.read_text(encoding='utf-8-sig'))
    size = _extract_frame_size(root)

    if size is None:
        print(f'  WARNING: no recognized size field on root frame of {figma_path} -- '
              f'using default viewport {DEFAULT_VIEWPORT}. Check the field name and extend '
              f'_extract_frame_size() if needed.')
        return dict(DEFAULT_VIEWPORT)

    return {'width': round(size['width']), 'height': round(size['height'])}

## 3. Inject generated stories into the existing GT file

In [90]:
def to_export_name(approach: str, prompt_strategy: str | None, variant: str | None, stem: str) -> str:
    """Builds a unique, JS-identifier-safe export name, e.g.
    's_b_zero_shot_b1_claude_sonnet_5_1' or, for uis, additionally prefixed
    with the pretty/messy variant.

    prompt_strategy is None for approach 'a' (rule-based/deterministic, no
    zero_shot/few_shot distinction) -- must be filtered out here rather than
    joined directly, or '_'.join() raises TypeError on the None."""
    parts = [approach]
    if prompt_strategy:
        parts.append(prompt_strategy)
    if variant:
        parts.append(variant)
    parts.append(stem)

    safe = '_'.join(parts).lower().replace('-', '_')
    safe = re.sub(r'[^a-z0-9_]', '_', safe)

    return safe


def relative_import_path(vue_file: Path, story_file: Path) -> str:
    rel = Path(__import__('os').path.relpath(vue_file, start=story_file.parent)).as_posix()
    return rel if rel.startswith('.') else f'./{rel}'


def inject_generated_stories(story_file: Path, entries: list[dict]) -> None:
    """entries: [{'import_var': str, 'import_path': str, 'export_name': str}, ...]
    Replaces the marker block if present, otherwise appends one.

    Matches the project's existing CSF2-style plain-object stories (no
    TypeScript type annotations, e.g. 'export const Default = {}') rather
    than a typed CSF3 'export const X: Story = {}' -- the latter would fail
    to compile here since these files never import or declare a 'Story'
    type at all.
    """
    if not story_file.exists():
        raise FileNotFoundError(
            f'Expected an existing GT story file at {story_file}, but it does not exist. '
            f'This script only edits existing stories, it does not create new ones.'
        )

    original = story_file.read_text(encoding='utf-8')

    if BEGIN_MARKER in original:
        pre = original.split(BEGIN_MARKER)[0].rstrip('\n') + '\n\n'
    else:
        pre = original.rstrip('\n') + '\n\n'

    block_lines = [BEGIN_MARKER]
    for e in entries:
        block_lines.append(f"import {e['import_var']} from '{e['import_path']}';")
    block_lines.append('')
    for e in entries:
        block_lines.append(
            f"export const {e['export_name']} = {{ "
            f"render: () => ({{ components: {{ {e['import_var']} }}, "
            f"template: '<{e['import_var']} />' }}) }};"
        )
    block_lines.append(END_MARKER)

    story_file.write_text(pre + '\n'.join(block_lines) + '\n', encoding='utf-8')


# Matches both plain CSF2 ('export const Default = {}') and typed CSF3
# ('export const X: Story = {}') -- the type annotation is optional so this
# works regardless of which style a given project (or even a given file)
# happens to use. Requires the line to actually be assigned an object/array/
# call (i.e. followed eventually by '='), not just any 'export const' use.
_STORY_EXPORT_RE = re.compile(
    r'^export\s+const\s+(\w+)\s*(?::\s*[\w.]+(?:<[^>]*>)?\s*)?=',
    re.MULTILINE,
)


def discover_story_export_names(content: str) -> list[str]:
    """Finds all top-level 'export const X (: Type)? = ...' declarations that
    exist BEFORE our auto-generated marker block, in order of appearance.
    Used to find the pre-existing GT story's own export name without
    assuming any particular naming convention (e.g. 'GT', 'Default', ...) --
    we do not know how the existing GT files name their story, so we read it
    back out of the file instead of guessing."""
    pre_marker = content.split(BEGIN_MARKER)[0] if BEGIN_MARKER in content else content

    return _STORY_EXPORT_RE.findall(pre_marker)

### 4. Main loop

In [91]:
gt_stories = find_gt_story_files()
print(f'Found {len(gt_stories)} existing GT story files under {STORIES_DIR}\n')

manifest: list[dict] = []
edited = 0
no_matches: list[str] = []
no_gt_export: list[str] = []

for gt in gt_stories:
    variants = find_generated_variants(gt['complexity'], gt['index'])

    label = f'{gt["complexity"]}/{gt["index"]}' if gt['complexity'] else gt['index']

    if not variants:
        no_matches.append(label)
        print(f'{label:20s}  0 generated variants found -- skipping (file left untouched)')
        continue

    # One viewport per mockup, shared by GT and every generated variant of it --
    # see '1b. Viewport per mockup' above for why this must not vary per variant.
    viewport = load_gt_viewport(gt['complexity'], gt['index'])

    # Manifest the PRE-EXISTING GT story too, so render_and_extract.py/ipynb also
    # captures a GT snapshot -- otherwise UF3 has nothing to compare generated
    # snapshots against. The export name is discovered from the file itself
    # (see discover_story_export_names()), not assumed.
    original_content = gt['story_file'].read_text(encoding='utf-8')
    gt_export_names = discover_story_export_names(original_content)

    if not gt_export_names:
        no_gt_export.append(label)
        print(f'  WARNING: no existing "export const X: Story" found in {gt["story_file"].name} -- '
              f'GT snapshot cannot be captured for this mockup.')
    else:
        if len(gt_export_names) > 1:
            print(f'  WARNING: multiple existing story exports found in {gt["story_file"].name} '
                  f'{gt_export_names} -- using the first one ({gt_export_names[0]!r}) as GT.')

        manifest.append({
            'storyFile': str(gt['story_file'].relative_to(REPO_ROOT)).replace('\\', '/'),
            'importPath': './' + str(gt['story_file'].relative_to(STORYBOOK_ROOT)).replace('\\', '/'),
            'exportName': gt_export_names[0],
            'approach': 'gt',
            'prompt_strategy': None,
            'complexity': gt['complexity'],
            'variant': None,
            'index': gt['index'],
            'stem': gt['index'],
            'vuePath': None,
            'viewportWidth': viewport['width'],
            'viewportHeight': viewport['height'],
        })

    entries = []
    for v in variants:
        export_name = to_export_name(v['approach'], v['prompt_strategy'], v['variant'], v['vue_file'].stem)
        import_var = 'Comp_' + export_name.removeprefix('s_')  # separate namespace from the export const itself
        import_path = relative_import_path(v['vue_file'], gt['story_file'])

        print(f'{label:20s}  {v["approach"]}/{v["prompt_strategy"] or "n/a"}/{v["variant"] or "n/a"} -> '
              f'{v["vue_file"].relative_to(REPO_ROOT)} -> {export_name}')

        entries.append({'import_var': import_var, 'import_path': import_path, 'export_name': export_name})

        manifest.append({
            'storyFile': str(gt['story_file'].relative_to(REPO_ROOT)).replace('\\', '/'),
            'importPath': './' + str(gt['story_file'].relative_to(STORYBOOK_ROOT)).replace('\\', '/'),
            'exportName': export_name,
            'approach': v['approach'],
            'prompt_strategy': v['prompt_strategy'],
            'complexity': gt['complexity'],
            'variant': v['variant'],
            'index': gt['index'],
            'stem': v['vue_file'].stem,
            'vuePath': str(v['vue_file'].relative_to(REPO_ROOT)).replace('\\', '/'),
            'viewportWidth': viewport['width'],
            'viewportHeight': viewport['height'],
        })

    inject_generated_stories(gt['story_file'], entries)
    edited += 1
    print(f'{label:20s}  {len(variants)} generated variants injected into {gt["story_file"].name}')

print(f'\nEdited {edited} existing story files, {len(manifest)} total manifest entries '
      f'(including {len(gt_stories) - len(no_gt_export) - len(no_matches)} GT entries).')
if no_matches:
    print(f'No generated variants found for: {no_matches}')
if no_gt_export:
    print(f'No GT export discovered for: {no_gt_export}')

Found 30 existing GT story files under C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\src\stories\components

hard/1                a/n/a/n/a -> dataset\storybook\src\code\components\a\hard\1-a.vue -> a_1_a
hard/1                b/few_shot/n/a -> dataset\storybook\src\code\components\b\few_shot\hard\1-b1-claude-sonnet-5-1.vue -> b_few_shot_1_b1_claude_sonnet_5_1
hard/1                b/few_shot/n/a -> dataset\storybook\src\code\components\b\few_shot\hard\1-b1-gemini-3.1-pro-preview-1.vue -> b_few_shot_1_b1_gemini_3_1_pro_preview_1
hard/1                b/few_shot/n/a -> dataset\storybook\src\code\components\b\few_shot\hard\1-b1-gpt-5.6-terra-1.vue -> b_few_shot_1_b1_gpt_5_6_terra_1
hard/1                b/few_shot/n/a -> dataset\storybook\src\code\components\b\few_shot\hard\1-b2-claude-sonnet-5-1.vue -> b_few_shot_1_b2_claude_sonnet_5_1
hard/1                b/few_shot/n/a -> dataset\storybook\src\code\components\b\few_shot\hard\1-b2-gemini-3.1-pro-preview-1.vue 

### 4. Save manifest

In [92]:
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'Manifest written: {MANIFEST_PATH} ({len(manifest)} entries)')

Manifest written: C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\evaluations\stories_manifest_components.json (1680 entries)
